In [1]:
import re
import unicodedata
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# ---------------- Config ----------------
JSONL_PATH = "HebNLI_balanced.jsonl"     # <-- put your file path here

TEXT1_COL = "translation1"
TEXT2_COL = "translation2"
LABEL_COL = "original_label"
GENRE_COL = "genre"
PAIR_COL = "pairID"

RANDOM_STATE = 42

# ---------------- Hebrew light normalization ----------------
HEBREW_NIQQUD_RE = re.compile(r"[\u0591-\u05C7]")  # cantillation + niqqud

def normalize_hebrew(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("״", '"').replace("׳", "'")
    s = s.replace("–", "-").replace("—", "-")
    s = HEBREW_NIQQUD_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ---------------- Load JSONL ----------------
df = pd.read_json(JSONL_PATH, lines=True)

needed = [TEXT1_COL, TEXT2_COL, LABEL_COL, GENRE_COL, PAIR_COL]
df = df[needed].copy()
df = df.dropna(subset=[TEXT1_COL, TEXT2_COL, LABEL_COL, GENRE_COL])

df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

allowed = {"entailment", "contradiction", "neutral"}
bad = set(df[LABEL_COL].unique()) - allowed
if bad:
    raise ValueError(f"Unexpected labels found: {bad}")

df["t1"] = df[TEXT1_COL].map(normalize_hebrew)
df["t2"] = df[TEXT2_COL].map(normalize_hebrew)

# Combine sentence pairs (for TF-IDF baseline)
df["combined"] = df["t1"] + " [SEP] " + df["t2"]

X = df["combined"].values
y = df[LABEL_COL].values

# ---------------- Split 70/15/15 ----------------
X_train, X_tmp, y_train, y_tmp, df_train, df_tmp = train_test_split(
    X, y, df, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

X_val, X_test, y_val, y_test, df_val, df_test = train_test_split(
    X_tmp, y_tmp, df_tmp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_tmp
)

print("Sizes:", len(X_train), len(X_val), len(X_test))
print("Train label counts:\n", pd.Series(y_train).value_counts())

# ---------------- Models ----------------
tfidf_char = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2)

pipelines = {
    "TFIDF(char3-5)+LogReg": Pipeline([
        ("tfidf", tfidf_char),
        ("clf", LogisticRegression(max_iter=3000))
    ]),
    "TFIDF(char3-5)+LinearSVM": Pipeline([
        ("tfidf", tfidf_char),
        ("clf", LinearSVC())
    ]),
}

def print_misclassified(df_te, y_true, y_pred, n=15):
    wrong = df_te.copy()
    wrong["y_true"] = y_true
    wrong["y_pred"] = y_pred
    wrong = wrong[wrong["y_true"] != wrong["y_pred"]]

    print(f"\nMisclassified examples (showing up to {n}): {len(wrong)} total\n")
    for _, row in wrong.head(n).iterrows():
        print("-" * 80)
        print(f"pairID: {row[PAIR_COL]} | genre: {row[GENRE_COL]}")
        print(f"TRUE: {row['y_true']} | PRED: {row['y_pred']}")
        print("t1:", row["t1"])
        print("t2:", row["t2"])

for name, model in pipelines.items():
    print("\n" + "=" * 80)
    print(name)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print(classification_report(y_test, preds, digits=4))
    print("Confusion matrix (rows=true, cols=pred) in label order [entailment, contradiction, neutral]:")
    print(confusion_matrix(y_test, preds, labels=["entailment", "contradiction", "neutral"]))

    # Required: error analysis examples
    print_misclassified(df_test, y_test, preds, n=20)

    # Nice for report: per-genre macro-F1
    print("\nPer-genre macro-F1 on test:")
    for g in sorted(df_test[GENRE_COL].unique()):
        mask = (df_test[GENRE_COL] == g).values
        if mask.sum() < 10:
            continue
        rep = classification_report(y_test[mask], preds[mask], output_dict=True, zero_division=0)
        print(f"  {g:12s}  n={mask.sum():4d}  macro_f1={rep['macro avg']['f1-score']:.4f}")


Sizes: 6732 1443 1443
Train label counts:
 entailment       2244
neutral          2244
contradiction    2244
Name: count, dtype: int64

TFIDF(char3-5)+LogReg
               precision    recall  f1-score   support

contradiction     0.3747    0.3451    0.3593       481
   entailment     0.2904    0.2952    0.2928       481
      neutral     0.3405    0.3617    0.3508       481

     accuracy                         0.3340      1443
    macro avg     0.3352    0.3340    0.3343      1443
 weighted avg     0.3352    0.3340    0.3343      1443

Confusion matrix (rows=true, cols=pred) in label order [entailment, contradiction, neutral]:
[[142 151 188]
 [166 166 149]
 [181 126 174]]

Misclassified examples (showing up to 20): 961 total

--------------------------------------------------------------------------------
pairID: 57267c | genre: travel
TRUE: contradiction | PRED: entailment
t1: חגיגות הארי ראיה פואסה או עיד אל-פיטר מציינות את סוף הרמדאן, או חודש הצום.
t2: חג הרי ראיה פואסה מציין את

In [4]:
!pip install scipy
#!/usr/bin/env python3

In [5]:
import pandas as pd
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# ---- Make sure these exist from your earlier notebook:
# df_train, df_test (dataframes)
# y_train, y_test (arrays)
# and that df_train has columns: t1, t2, genre, pairID

# ---------------- Feature builder: compare t1 vs t2 ----------------
class PairTfidfFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, analyzer="char_wb", ngram_range=(3, 5), min_df=2):
        self.analyzer = analyzer
        self.ngram_range = ngram_range
        self.min_df = min_df
        self.vec = TfidfVectorizer(
            analyzer=self.analyzer,
            ngram_range=self.ngram_range,
            min_df=self.min_df
        )

    def fit(self, X, y=None):
        # Fit vectorizer on BOTH sides together
        all_text = pd.concat([X["t1"], X["t2"]], axis=0).astype(str)
        self.vec.fit(all_text)
        return self

    def transform(self, X):
        v1 = self.vec.transform(X["t1"].astype(str))
        v2 = self.vec.transform(X["t2"].astype(str))

        diff = abs(v1 - v2)          # how different
        prod = v1.multiply(v2)       # how much overlap

        # final features: [v1, v2, |v1-v2|, v1*v2]
        return sparse.hstack([v1, v2, diff, prod], format="csr")

# ---------------- Model ----------------
model = Pipeline([
    ("feats", PairTfidfFeatures(analyzer="char_wb", ngram_range=(3, 5), min_df=2)),
    ("clf", LinearSVC())
])

# ---------------- Train ----------------
X_train_pairs = df_train[["t1", "t2"]]
X_test_pairs  = df_test[["t1", "t2"]]

model.fit(X_train_pairs, y_train)

# ---------------- Predict ----------------
preds = model.predict(X_test_pairs)

# ---------------- Print results ----------------
print(classification_report(y_test, preds, digits=4))
print("Confusion matrix (rows=true, cols=pred) label order [entailment, contradiction, neutral]:")
print(confusion_matrix(y_test, preds, labels=["entailment", "contradiction", "neutral"]))

# ---------------- Per-genre macro-F1 ----------------
print("\nPer-genre macro-F1 on test:")
for g in sorted(df_test["genre"].unique()):
    mask = (df_test["genre"] == g).values
    rep = classification_report(y_test[mask], preds[mask], output_dict=True, zero_division=0)
    print(f"  {g:12s}  n={mask.sum():4d}  macro_f1={rep['macro avg']['f1-score']:.4f}")

# ---------------- Show some wrong examples (error analysis) ----------------
wrong = df_test.copy()
wrong["y_true"] = y_test
wrong["y_pred"] = preds
wrong = wrong[wrong["y_true"] != wrong["y_pred"]]

print(f"\nMisclassified examples: {len(wrong)} total. Showing 15:\n")
for _, row in wrong.head(15).iterrows():
    print("-" * 80)
    print(f"pairID: {row['pairID']} | genre: {row['genre']}")
    print(f"TRUE: {row['y_true']} | PRED: {row['y_pred']}")
    print("t1:", row["t1"])
    print("t2:", row["t2"])


               precision    recall  f1-score   support

contradiction     0.4522    0.4324    0.4421       481
   entailment     0.3810    0.4096    0.3948       481
      neutral     0.4163    0.4033    0.4097       481

     accuracy                         0.4151      1443
    macro avg     0.4165    0.4151    0.4155      1443
 weighted avg     0.4165    0.4151    0.4155      1443

Confusion matrix (rows=true, cols=pred) label order [entailment, contradiction, neutral]:
[[197 127 157]
 [158 208 115]
 [162 125 194]]

Per-genre macro-F1 on test:
  fiction       n= 217  macro_f1=0.4693
  government    n= 217  macro_f1=0.4728
  letters       n= 199  macro_f1=0.4205
  nineeleven    n= 190  macro_f1=0.3369
  oup           n= 207  macro_f1=0.3119
  slate         n= 213  macro_f1=0.4059
  travel        n= 200  macro_f1=0.4749

Misclassified examples: 844 total. Showing 15:

--------------------------------------------------------------------------------
pairID: 57267c | genre: travel
TRUE: 

In [8]:
import pandas as pd
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# ---- Make sure these exist from your earlier notebook:
# df_train, df_test (dataframes)
# y_train, y_test (arrays)
# and that df_train has columns: t1, t2, genre, pairID

# ---------------- Feature builder: compare t1 vs t2 ----------------
class PairTfidfFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, analyzer="char_wb", ngram_range=(2, 6), min_df=2):
        self.analyzer = analyzer
        self.ngram_range = ngram_range
        self.min_df = min_df
        self.vec = TfidfVectorizer(
            analyzer=self.analyzer,
            ngram_range=self.ngram_range,
            min_df=self.min_df
        )

    def fit(self, X, y=None):
        # Fit vectorizer on BOTH sides together
        all_text = pd.concat([X["t1"], X["t2"]], axis=0).astype(str)
        self.vec.fit(all_text)
        return self

    def transform(self, X):
        v1 = self.vec.transform(X["t1"].astype(str))
        v2 = self.vec.transform(X["t2"].astype(str))

        diff = abs(v1 - v2)          # how different
        prod = v1.multiply(v2)       # how much overlap

        # final features: [v1, v2, |v1-v2|, v1*v2]
        return sparse.hstack([v1, v2, diff, prod], format="csr")

# ---------------- Model ----------------
model = Pipeline([
    ("feats", PairTfidfFeatures(analyzer="char_wb", ngram_range=(2, 6), min_df=2)),
    ("clf", LinearSVC())
])

# ---------------- Train ----------------
X_train_pairs = df_train[["t1", "t2"]]
X_test_pairs  = df_test[["t1", "t2"]]

model.fit(X_train_pairs, y_train)

# ---------------- Predict ----------------
preds = model.predict(X_test_pairs)

# ---------------- Print results ----------------
print(classification_report(y_test, preds, digits=4))
print("Confusion matrix (rows=true, cols=pred) label order [entailment, contradiction, neutral]:")
print(confusion_matrix(y_test, preds, labels=["entailment", "contradiction", "neutral"]))

# ---------------- Per-genre macro-F1 ----------------
print("\nPer-genre macro-F1 on test:")
for g in sorted(df_test["genre"].unique()):
    mask = (df_test["genre"] == g).values
    rep = classification_report(y_test[mask], preds[mask], output_dict=True, zero_division=0)
    print(f"  {g:12s}  n={mask.sum():4d}  macro_f1={rep['macro avg']['f1-score']:.4f}")

# ---------------- Show some wrong examples (error analysis) ----------------
wrong = df_test.copy()
wrong["y_true"] = y_test
wrong["y_pred"] = preds
wrong = wrong[wrong["y_true"] != wrong["y_pred"]]

print(f"\nMisclassified examples: {len(wrong)} total. Showing 15:\n")
for _, row in wrong.head(15).iterrows():
    print("-" * 80)
    print(f"pairID: {row['pairID']} | genre: {row['genre']}")
    print(f"TRUE: {row['y_true']} | PRED: {row['y_pred']}")
    print("t1:", row["t1"])
    print("t2:", row["t2"])


               precision    recall  f1-score   support

contradiction     0.4602    0.4324    0.4459       481
   entailment     0.4141    0.4407    0.4270       481
      neutral     0.4447    0.4428    0.4437       481

     accuracy                         0.4387      1443
    macro avg     0.4396    0.4387    0.4389      1443
 weighted avg     0.4396    0.4387    0.4389      1443

Confusion matrix (rows=true, cols=pred) label order [entailment, contradiction, neutral]:
[[212 122 147]
 [154 208 119]
 [146 122 213]]

Per-genre macro-F1 on test:
  fiction       n= 217  macro_f1=0.4888
  government    n= 217  macro_f1=0.5107
  letters       n= 199  macro_f1=0.4419
  nineeleven    n= 190  macro_f1=0.3463
  oup           n= 207  macro_f1=0.3387
  slate         n= 213  macro_f1=0.4424
  travel        n= 200  macro_f1=0.4844

Misclassified examples: 810 total. Showing 15:

--------------------------------------------------------------------------------
pairID: 57267c | genre: travel
TRUE: 